In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

import tensorflow as tf

Now import the dataset and check it

1. I  used custom column names
2. Read the csv file, skipped the headers
3. iloc is used to skip the first row (because it wasn't removing the previous column names)
4. head() print first five rows




In [ ]:
cols = ["timestamp", "ac-stage","peer-pressure" ,"home-pressure", "study-env", "coping-statergy", "bad-habits","rate-acedemic-comptetiton", "rate-stress"]
df = pd.read_csv("academicStressLevel.csv", names=cols,header=None)
df = df.iloc[1:]

1. to_numeric is used to convert into numeric type, coerce is used to handle the errors and enter NaN.
2. NaN values removed with dropna
3. converted to int

In [ ]:
df["rate-stress"] = pd.to_numeric(df["rate-stress"],errors='coerce')
df.dropna(subset=['rate-stress'], inplace=True)
df["rate-stress"] = df["rate-stress"].astype(int)

I want to create a new column named as is_stressed. If student is having greater than 3 stress rate it is cocnsidered as stressesd
1. we can do either way numpy or lambda
2. numpy preffered

In [ ]:
stress_threshold = 3
# df["is_stressed"] = np.where(df["rate-stress"] >= stress_threshold, 'Yes','No')
df["is_stressed"]  = df["rate-stress"].apply(lambda x: 'Yes' if x >= stress_threshold else 'No')

Now change yes or no column with your 0,1 as it is easier for machine

In [ ]:
df["is_stressed"] = (df["is_stressed"] == "Yes").astype(int)
# df.head()

Plot a graph of every column

In [ ]:
numerical_cols = ["home-pressure", "peer-pressure", "rate-acedemic-comptetiton"]
# for label in numerical_cols:
  # plt.figure()
  # plt.hist(df[df["is_stressed"] == 1][label], color = 'red', label="stressed" ,alpha=0.6,density=True)
  # plt.hist(df[df["is_stressed"] == 1][label], color = 'green', label = "not stressed" ,alpha=0.6,density=True)
  # plt.title(label)
  # plt.xlabel(label)
  # plt.ylabel("Density")
  # plt.legend()
  # plt.grid(axis='y', alpha=0.75)
  # plt.tight_layout()
  # plt.show()

In [ ]:
# df.head()

Train, validation, test datasets

In [ ]:
train,valid, test = np.split(df.sample(frac=1), [int(len(df) * 0.6), int(0.8 * len(df))])

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Create a function to scale and preprocess the dataset
This function processes a DataFrame by separating features and target,
    applying appropriate scaling and encoding, and optionally oversampling.

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        oversample (bool): Whether to apply RandomOverSampler to the data.

    Returns:
        tuple: A tuple containing the processed features (X), and target (y).
Create a preprocessor using ColumnTransformer.

    This applies StandardScaler to numerical columns and OneHotEncoder to categorical columns.
    Keep other columns as is (like timestamp if we included it)

ColumnTransformer from scikit-learn is used, which allows us to apply different preprocessing steps to different columns simultaneously. We will use StandardScaler for the numerical columns and OneHotEncoder for the categorical ones.

The RandomOverSampler part is also very important. It addresses class imbalance by creating synthetic samples for the minority class, ensuring your model doesn't become biased towards the more common class

Below code can be used to check the data

In [ ]:
print(len(train[train['is_stressed'] == 1]))
print(len(train[train['is_stressed'] == 0]))

77
7


In [ ]:
def scale_dataset(dataframe, preprocessor=None, overSample=False):
  # Explicitly drop timestamp and any other non-numeric columns not in categorical_cols
  cols_to_drop = ['timestamp']
  for col in dataframe.columns:
    if dataframe[col].dtype == 'object' and col not in ['is_stressed'] + ['ac-stage', 'study-env', 'coping-statergy','bad-habits']:
        cols_to_drop.append(col)

  X = dataframe.drop(columns=cols_to_drop).copy()
  y = dataframe['is_stressed'].copy()


  numerical_cols = ['home-pressure', 'rate-acedemic-comptetiton', 'rate-stress']
  categorical_cols = ['ac-stage', 'study-env', 'coping-statergy','bad-habits']

  # Ensure numerical_cols and categorical_cols only contain columns present in X
  numerical_cols = [col for col in numerical_cols if col in X.columns]
  categorical_cols = [col for col in categorical_cols if col in X.columns]


  if preprocessor is None:
    preprocessor = ColumnTransformer(
        transformers=[
            ('num',StandardScaler(), numerical_cols),
            ('cat',OneHotEncoder(handle_unknown='ignore'),categorical_cols)
        ],
        remainder = 'passthrough' # Keep this in case there are unexpected numerical columns
    )
    X_processed = preprocessor.fit_transform(X)
  else:
    X_processed = preprocessor.transform(X)

  # Convert sparse matrix to dense array if necessary
  if hasattr(X_processed, 'toarray'):
      X_processed = X_processed.toarray()


  if overSample:
    unique, counts = np.unique(y, return_counts=True)
    class_distribution = dict(zip(unique,counts))
    print(f"Original Class Distribution: {class_distribution}\n")

    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X_processed,y)

    unique_resampled, counts_resampled = np.unique(y_resampled, return_counts=True)
    resampled_distribution = dict(zip(unique_resampled, counts_resampled))
    print(f"Resampled class distribution: {resampled_distribution}\n")

    return X_resampled, y_resampled, preprocessor

  return X_processed, y, preprocessor

In [ ]:
print(len(train[train['is_stressed'] == 1]))
print(len(train[train['is_stressed'] == 0]))

77
7


In [ ]:
X_train, y_train, preprocessor = scale_dataset(train, overSample=True)
X_valid, y_valid, _ = scale_dataset(valid, preprocessor=preprocessor, overSample=False)
X_test, y_test, _ = scale_dataset(test, preprocessor=preprocessor, overSample=False)

Original Class Distribution: {np.int64(0): np.int64(7), np.int64(1): np.int64(77)}

Resampled class distribution: {np.int64(0): np.int64(77), np.int64(1): np.int64(77)}



KNN Algorithm
1. Choose value of K
2. Calculate Distance
3. Find K-Nearest Neighbors
4. Vote for a class

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)
# print(confusion_matrix(y_test, y_pred))
# print("Accuracy: ", accuracy_score(y_test, y_pred))
# print("Precision: ", precision_score(y_test, y_pred))
# print("Recall: ", recall_score(y_test, y_pred))
# print("F1 Score: ", f1_score(y_test, y_pred))

Naive_Bayes Algorithm

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)
# print(confusion_matrix(y_test, y_pred))
# print("Accuracy: ", accuracy_score(y_test, y_pred))
# print("Precision: ", precision_score(y_test, y_pred))
# print("Recall: ", recall_score(y_test, y_pred))
# print("F1 Score: ", f1_score(y_test, y_pred))

Logistic Regression

In [ ]:
lg_model = LogisticRegression()
lg_model.fit(X_train, y_train)
y_pred = lg_model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))
print(classification_report(y_test,y_pred))

[[ 5  0]
 [ 0 23]]
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
F1 Score:  1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00        23

    accuracy                           1.00        28
   macro avg       1.00      1.00      1.00        28
weighted avg       1.00      1.00      1.00        28



Support Vector Machine

In [ ]:
svm_model = SVC()
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))
print(classification_report(y_test,y_pred))

[[ 5  0]
 [ 0 23]]
Accuracy:  1.0
Precision:  1.0
Recall:  1.0
F1 Score:  1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00        23

    accuracy                           1.00        28
   macro avg       1.00      1.00      1.00        28
weighted avg       1.00      1.00      1.00        28



Neural Networks
1. Activation function - introduces non-linearity to model
2. Backpropogation - repeat
3. Activation Functions- RelU, Tanh, Sigmoid, Softmax
4. TenserFlow - develop and train models


In [ ]:
def plot_loss(history):
  plt.plot(history.history['loss'], label='Training Loss')
  plt.plot(history.history['val_loss'], label='Validation Loss')
  plt.title('Loss over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()
  plt.show()

def plot_accuracy(history):
  plt.plot(history.history['accuracy'], label='Training Accuracy')
  plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
  plt.title('Accuracy over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.legend()
  plt.show()

In [ ]:
nn_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64,activation='relu',input_shape=(X_train.shape[1],),),
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])
nn_model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
history= nn_model.fit(X_train,y_train,epochs=100, batch_size=32, validation_split=0.2, verbose=0)

In [ ]:
# plot_loss(history)
# plot_accuracy(history)

In [ ]:
y_pred = nn_model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step


In [ ]:
y_pred.reshape(-1)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00        23

    accuracy                           1.00        28
   macro avg       1.00      1.00      1.00        28
weighted avg       1.00      1.00      1.00        28



Linear Regression